# QuizForge - fine-tune Qwen 2.5 (3B and 7B) for MCQ generation

This notebook fine-tunes both models for exactly the task the app gives them: *write multiple-choice questions as JSON
from source material*. It uses the app's own system prompt, prompt builder and validator, so the training data matches
what the app sends and accepts. It runs on **Kaggle** or **Google Colab** (detected automatically).

**Steps** (about 2-3 hours on a T4 GPU):
1. Settings, GPU and internet check, project code, Unsloth install (checked before anything long starts).
2. Download SciQ (a public science-MCQ dataset with source paragraphs) and convert it to the app's format.
3. *Teacher data*: run the app's real generator with `qwen2.5:7b` (through Ollama) on held-out SciQ passages and keep only
   questions the validator accepts **and** `qwen2.5:3b` answers the same way blind.
4. Build `train.jsonl` / `val.jsonl`.
5. QLoRA fine-tuning of the 3B model, then of the 7B model; each is exported as a q4_K_M GGUF file.
6. Download the GGUF files and run `finetune\import_finetuned.bat` on your laptop.

**Kaggle**
1. Right-hand panel -> *Settings*: *Accelerator* -> **GPU T4 x2**, *Internet* -> **On** (needs a verified phone number).
2. **Quick test first** (~30 min): set `QUICK_TEST = True` in the first cell and *Run All*. It runs every step on a tiny
   dataset, including the GGUF export, so a problem shows up in minutes rather than after hours.
3. **Real run**: set `QUICK_TEST = False`, then *Save Version* -> **Save & Run All (Commit)**. It runs in the background
   (you can close the browser) and the finished files stay in that version's *Output* tab.

**Colab:** *Runtime -> Change runtime type -> T4 GPU*, then *Run all*. Finished files are copied to Google Drive
(`MyDrive/quizforge_finetune`).

Data licence: SciQ is CC BY-NC 3.0 (non-commercial use). Base models: Qwen2.5-3B-Instruct (Qwen Research Licence,
non-commercial) and Qwen2.5-7B-Instruct (Apache 2.0).

## 1. Settings, GPU, internet and project code

In [ ]:
# ---- settings ------------------------------------------------------------------------------------------------
QUICK_TEST = False      # True: tiny run (~30 min, 3B only) that checks every step incl. the GGUF export
TRAIN_7B = True         # False: fine-tune only the 3B model
TEACHER_PASSAGES = 2 if QUICK_TEST else 80    # 0 skips the teacher step
SCIQ_EXAMPLES = 60 if QUICK_TEST else 1800
CHECK_PROMPTS = 3 if QUICK_TEST else 20       # validation prompts for the format check
SUFFIX = "-test" if QUICK_TEST else ""        # quick-test models are named quizforge-3b-test
# ----------------------------------------------------------------------------------------------------------------

import os, sys, shutil, subprocess, json, glob
ON_KAGGLE = os.path.exists("/kaggle/working")
PLATFORM = "Kaggle" if ON_KAGGLE else "Colab"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Kaggle's "T4 x2" has two GPUs; Unsloth trains on one

if not shutil.which("nvidia-smi"):
    raise SystemExit("No GPU in this session.\n  Kaggle: Settings -> Accelerator -> GPU T4 x2\n"
                     "  Colab: Runtime -> Change runtime type -> T4 GPU\nThen run the notebook again from the top.")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

# WORK: a large scratch disk for code, data, checkpoints and the GGUF conversion (merging the 7B model needs ~35 GB at peak).
# Kaggle's /kaggle/working is capped at 20 GB, so the work happens in /tmp and only finished files go to /kaggle/working.
WORK = "/tmp/quizforge" if ON_KAGGLE else "/content/quizforge"
free_gb = shutil.disk_usage("/tmp" if ON_KAGGLE else "/content").free / 1e9
print(f"Platform: {PLATFORM}, Python {sys.version.split()[0]}, work folder: {WORK}, free disk: {free_gb:.0f} GB")
print("QUICK TEST - tiny run, 3B only" if QUICK_TEST else "Full run: 3B" + (" and 7B" if TRAIN_7B else " only"))
if free_gb < 40:
    print("WARNING: less than 40 GB free - the 7B GGUF export may run out of disk.")

Option A clones the GitHub repository. Option B: run `.venv\Scripts\python.exe -m finetune.make_colab_bundle` on your
laptop and upload `quizforge_colab.zip` (Kaggle: *Add Input* -> *Upload* as a dataset; Colab: the file browser on the left),
then set `USE_ZIP = True`. Both options need internet for the packages, SciQ and the base models.

In [ ]:
import zipfile, socket
USE_ZIP = False
REPO = "https://github.com/Suketu-ADT/quizforge-mcq-generator.git"

def online(host):
    try:
        socket.create_connection((host, 443), timeout=5).close()
        return True
    except OSError:
        return False

# The notebook needs the internet in every option: pip (Unsloth), Hugging Face (SciQ, base models), GitHub (code)
offline = [h for h in ("github.com", "pypi.org", "huggingface.co") if not online(h)]
if offline:
    raise SystemExit(
        "No internet in this session (cannot reach " + ", ".join(offline) + ").\n"
        + ("Kaggle: right-hand panel -> Settings -> Internet -> On.\n"
           "  Greyed out? Verify your phone number: kaggle.com -> profile picture -> Settings -> Phone verification, then reload.\n"
           "  If Kaggle restarts the session, run the notebook again from the FIRST cell."
           if ON_KAGGLE else "Check the Colab runtime's connection, then run this cell again."))

if not os.path.exists(f"{WORK}/finetune/train.py"):
    shutil.rmtree(WORK, ignore_errors=True)   # remove a half-finished clone from an earlier failed attempt
    if USE_ZIP:
        zips = glob.glob("/kaggle/input/**/quizforge_colab.zip", recursive=True) + glob.glob("/content/quizforge_colab.zip")
        assert zips, "quizforge_colab.zip not found - upload it first"
        zipfile.ZipFile(zips[0]).extractall(WORK)
    else:
        done = subprocess.run(["git", "clone", "--depth", "1", REPO, WORK], capture_output=True, text=True)
        if done.returncode:
            raise SystemExit("git clone failed:\n" + done.stderr)
os.chdir(WORK)
assert os.path.exists("finetune/train.py"), "finetune/ is missing - push it to GitHub or use the zip (option B)"
print("project ready:", os.getcwd())

Installs Unsloth (it brings its own compatible torch, TRL, transformers and datasets) and checks it in a separate process,
so a broken install stops here instead of after the teacher step.

In [ ]:
!pip -q install unsloth python-dotenv
check = subprocess.run([sys.executable, "-c",
    "import torch, trl, transformers, unsloth\n"
    "from unsloth.chat_templates import get_chat_template, train_on_responses_only\n"
    "print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '| unsloth', unsloth.__version__,"
    " '| trl', trl.__version__, '| transformers', transformers.__version__)"], capture_output=True, text=True)
print(check.stdout.strip().splitlines()[-1] if check.stdout.strip() else "")
if check.returncode or "CUDA True" not in check.stdout:
    raise SystemExit("Unsloth install check failed - the training would fail too:\n" + check.stderr[-3000:])

In [ ]:
# Where finished files go: Kaggle -> /kaggle/working (Output panel); Colab -> Google Drive
OUT = f"{WORK}/finetune"
if ON_KAGGLE:
    DELIVER = "/kaggle/working"
else:
    from google.colab import drive
    drive.mount("/content/drive")
    DELIVER = "/content/drive/MyDrive/quizforge_finetune"
os.makedirs(DELIVER, exist_ok=True)
print("finished files go to:", DELIVER)

def save(path):
    if os.path.exists(path):
        shutil.copy(path, DELIVER)
        print("saved:", os.path.join(DELIVER, os.path.basename(path)))
    else:
        print("not found (skipped):", path)

def deliver_model(name):
    """Copy a trained model's report, GGUF file and (as a small backup) its LoRA adapter to DELIVER."""
    exports = f"{OUT}/exports"
    save(f"{exports}/{name}-report.json")
    save(f"{exports}/{name}.Q4_K_M.gguf")
    if os.path.isdir(f"{exports}/{name}-lora"):
        print("saved:", shutil.make_archive(f"{DELIVER}/{name}-lora", "zip", f"{exports}/{name}-lora"))
    report = f"{exports}/{name}-report.json"
    if not os.path.exists(report):
        print(f"!! {name}: training did not finish - read the log above.")
    elif "gguf_error" in json.load(open(report)):
        print(f"!! {name}: training finished but the GGUF export failed - the adapter zip above keeps the result.")

## 2. SciQ
11,679 training rows: a question, the correct answer, three distractors and the paragraph that supports it.
The training split becomes SciQ examples; the validation split only supplies passages for the teacher step, so the two
never share a paragraph.

In [ ]:
from datasets import load_dataset
sciq = load_dataset("allenai/sciq")
os.makedirs(f"{OUT}/data", exist_ok=True)
for split in ("train", "validation"):
    with open(f"{OUT}/data/sciq_{split}.jsonl", "w", encoding="utf-8") as f:
        for row in sciq[split]:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(split, len(sciq[split]))

## 3. Teacher data (optional but recommended, ~30-45 min)
Installs Ollama inside this machine, pulls the two base models and runs `finetune/distill.py`.
Set `TEACHER_PASSAGES = 0` in the first cell to skip this step.

In [ ]:
if TEACHER_PASSAGES:
    !apt-get -qq install -y zstd > /dev/null
    !curl -fsSL https://ollama.com/install.sh | sh > /dev/null
    import time, requests
    server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(30):
        try:
            requests.get("http://localhost:11434/api/version", timeout=2); break
        except Exception:
            time.sleep(1)
    !ollama pull qwen2.5:7b
    !ollama pull qwen2.5:3b

In [ ]:
if TEACHER_PASSAGES:
    !{sys.executable} -m finetune.distill --rows finetune/data/sciq_validation.jsonl --out finetune/data/distilled.jsonl --passages {TEACHER_PASSAGES}
    save(f"{OUT}/data/distilled.jsonl")
    server.terminate()                       # free the GPU for training
    !pkill -f "ollama" || true
    !rm -rf ~/.ollama/models                 # free ~7 GB of disk for the GGUF export

## 4. Training data

In [ ]:
!{sys.executable} -m finetune.build_dataset --sciq finetune/data/sciq_train.jsonl --distilled finetune/data/distilled.jsonl --out finetune/data --sciq-examples {SCIQ_EXAMPLES}
for f in ("train.jsonl", "val.jsonl", "dataset_info.json"):
    save(f"{OUT}/data/{f}")

In [ ]:
# look at one training example
ex = json.loads(open(f"{OUT}/data/train.jsonl", encoding="utf-8").readline())
print(ex["messages"][1]["content"][-700:])
print("\n--- target ---\n", ex["messages"][2]["content"][:900])

## 5a. Fine-tune the 3B model (~20-30 min incl. GGUF export)

In [ ]:
NAME_3B = "quizforge-3b" + SUFFIX
!{sys.executable} -m finetune.train --base unsloth/Qwen2.5-3B-Instruct-bnb-4bit --name {NAME_3B} --data finetune/data --out finetune/exports --check {CHECK_PROMPTS}
deliver_model(NAME_3B)
!rm -rf ~/.cache/huggingface/hub/models--unsloth--Qwen2.5-3B*   # free disk before the 7B run

## 5b. Fine-tune the 7B model (~45-70 min incl. GGUF export)
Uses batch size 1 with 8 accumulation steps to fit a T4's 16 GB. Skipped in the quick test.

In [ ]:
if QUICK_TEST or not TRAIN_7B:
    print("7B skipped (" + ("quick test" if QUICK_TEST else "TRAIN_7B = False") + ")")
else:
    print(f"free disk: {shutil.disk_usage(WORK).free / 1e9:.0f} GB")
    !{sys.executable} -m finetune.train --base unsloth/Qwen2.5-7B-Instruct-bnb-4bit --name quizforge-7b --data finetune/data --out finetune/exports --batch 1 --grad-accum 8 --check {CHECK_PROMPTS}
    deliver_model("quizforge-7b")

## 6. Results and download
The reports show the validation loss before and after training and the format check (how many fine-tuned replies pass
the app's validator **without** a JSON-schema constraint).

- **Kaggle:** the version's *Output* tab (or the right-hand *Output* panel in an interactive session) -> download
  `quizforge-3b.Q4_K_M.gguf` (~2 GB), `quizforge-7b.Q4_K_M.gguf` (~4.7 GB) and the two `-report.json` files.
  In an interactive session, download them **before** ending it.
- **Colab:** download them from the Drive folder `quizforge_finetune`.

Put the `.gguf` files in `finetune\exports\` on your laptop and double-click `finetune\import_finetuned.bat`, then
`finetune\compare_finetuned.bat` compares each fine-tuned model with its base model on all three tests.
The `-lora.zip` files are a small backup of the trained adapters (needed only if a GGUF export failed).

In [ ]:
for name in ("quizforge-3b" + SUFFIX, "quizforge-7b"):
    p = f"{OUT}/exports/{name}-report.json"
    if os.path.exists(p):
        r = json.load(open(p))
        print(name, {k: r.get(k) for k in ("train_examples", "train_loss", "val_loss_before", "val_loss_after", "gguf_gb")})
        print("  format check:", r.get("format_check"))
        if r.get("gguf_error"):
            print("  GGUF export failed:", r["gguf_error"])
print("\nfiles ready for download in", DELIVER)
for f in sorted(os.listdir(DELIVER)):
    p = os.path.join(DELIVER, f)
    if os.path.isfile(p):
        print(f"  {f:40} {os.path.getsize(p) / 1e9:6.2f} GB")
if QUICK_TEST:
    print("\nQuick test finished. If both steps above look right, set QUICK_TEST = False and use Save Version -> Save & Run All.")